In [1]:
#Install dependencies
!pip install -q ultralytics opencv-python pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.9 MB/s eta 0:00:00


In [2]:
import os
import yaml
import cv2
from google.colab import files
from IPython.display import HTML
from base64 import b64encode

import ultralytics
from ultralytics.data.utils import check_det_dataset
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Download COCO via Ultralytics and get paths

# Locate built-in coco.yaml and download missing files
coco_yaml = os.path.join(ultralytics.__path__[0], 'cfg', 'datasets', 'coco.yaml')
data_info = check_det_dataset(coco_yaml, autodownload=True)

print(f"COCO root: {data_info['path']}")
print(f" Train:   {data_info['train']}")
print(f" Val:     {data_info['val']}")



WARNING ⚠️ Dataset '/usr/local/lib/python3.11/dist-packages/ultralytics/cfg/datasets/coco.yaml' images not found, missing path '/content/datasets/coco/val2017.txt'


100%|██████████| 169M/169M [00:00<00:00, 248MB/s]
Unzipping /content/datasets/coco2017labels-segments.zip to /content/datasets/coco...: 100%|██████████| 122232/122232 [00:16<00:00, 7474.01file/s]

Dataset download success ✅ (280.1s), saved to /content/datasets



100%|██████████| 755k/755k [00:00<00:00, 14.4MB/s]

COCO root: /content/datasets/coco
 Train:   /content/datasets/coco/train2017.txt
 Val:     /content/datasets/coco/val2017.txt


In [ ]:
# !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/datasets/coco/annotations/anno.zip

In [ ]:
# !unzip -q /content/datasets/coco/annotations/anno.zip -d /content/datasets/coco/annotations

In [4]:
# Build traffic.yaml with string paths
import yaml

TRAFFIC_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle',
    'bus', 'truck', 'traffic light', 'stop sign'
]

# Ensure all Path objects are converted to strings
traffic_spec = {
    'path':  str(data_info['path']),
    'train': str(data_info['train']),
    'val':   str(data_info['val']),
    'nc':    len(TRAFFIC_CLASSES),
    'names': TRAFFIC_CLASSES
}

with open('traffic.yaml', 'w') as f:
    yaml.safe_dump(traffic_spec, f, sort_keys=False)

print("▶ traffic.yaml created:")
print(yaml.safe_dump(traffic_spec, sort_keys=False))


▶ traffic.yaml created:
path: /content/datasets/coco
train: /content/datasets/coco/train2017.txt
val: /content/datasets/coco/val2017.txt
nc: 8
names:
- person
- bicycle
- car
- motorcycle
- bus
- truck
- traffic light
- stop sign



In [6]:
# Train YOLOv8 on traffic-only COCO

model = YOLO('yolov8n.pt')  # or yolov8s.pt / yolov8m.pt

model.train(
    data='traffic.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    project='runs/traffic',
    name='yolov8n_traffic'
)


Ultralytics 8.3.127 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=traffic.yaml, degrees=0.0, deterministic=True, device=cuda:0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_traffic2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

train: Scanning /content/datasets/coco/labels/train2017... 117266 images, 1021 backgrounds, 107570 corrupt: 100%|██████████| 118287/118287 [01:28<00:00, 1338.29it/s]


Streaming output truncated to the last 5000 lines.
train: /content/datasets/coco/images/train2017/000000555582.jpg: ignoring corrupt image/label: Label class 26 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/datasets/coco/images/train2017/000000555583.jpg: ignoring corrupt image/label: Label class 36 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/datasets/coco/images/train2017/000000555586.jpg: ignoring corrupt image/label: Label class 75 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/datasets/coco/images/train2017/000000555590.jpg: ignoring corrupt image/label: Label class 67 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/datasets/coco/images/train2017/000000555602.jpg: ignoring corrupt image/label: Label class 65 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/datasets/coco/images/train2017/000000555606.jpg: ignoring corrupt image/label:

val: Scanning /content/datasets/coco/labels/val2017... 4952 images, 48 backgrounds, 4545 corrupt: 100%|██████████| 5000/5000 [00:09<00:00, 534.49it/s]

val: /content/datasets/coco/images/val2017/000000000139.jpg: ignoring corrupt image/label: Label class 75 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/datasets/coco/images/val2017/000000000285.jpg: ignoring corrupt image/label: Label class 21 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/datasets/coco/images/val2017/000000000632.jpg: ignoring corrupt image/label: Label class 73 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/datasets/coco/images/val2017/000000000724.jpg: ignoring corrupt image/label: Label class 11 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/datasets/coco/images/val2017/000000000776.jpg: ignoring corrupt image/label: Label class 77 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/datasets/coco/images/val2017/000000000785.jpg: ignoring corrupt image/label: Label class 30 exceeds dataset class count 8. Possible class labels are 0-

...

      10/10      2.83G      1.159      1.093      1.204         51        640: 100%|██████████| 670/670 [01:00<00:00, 11.04it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:01<00:00,  7.95it/s]


                   all        455       1935      0.713       0.57      0.657      0.457

10 epochs completed in 0.178 hours.
Optimizer stripped from runs/traffic/yolov8n_traffic2/weights/last.pt, 6.2MB
Optimizer stripped from runs/traffic/yolov8n_traffic2/weights/best.pt, 6.2MB

Validating runs/traffic/yolov8n_traffic2/weights/best.pt...
Ultralytics 8.3.127 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:03<00:00,  4.59it/s]


                   all        455       1935      0.715      0.569      0.657      0.456
                person        201        773      0.686      0.459      0.549      0.313
               bicycle         24         60      0.684      0.383      0.517      0.277
                   car        108        355      0.674      0.493      0.536      0.346
            motorcycle         81        195      0.726      0.574      0.688      0.436
                   bus         84        123      0.891      0.795      0.887      0.694
                 truck         91        143      0.706      0.713      0.769       0.64
         traffic light         88        112      0.767      0.768       0.83      0.634
             stop sign         86        174      0.584      0.363      0.479       0.31
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 1.3ms postprocess per image
Saving runs/traffic/yolov8n_traffic2/predictions.json...

Evaluating pycocotools mAP using runs/traffic/yolov8n_traff

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dfb15f26890>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [ ]:
# Upload & process a video file

# 1. Upload
uploaded = files.upload()
video_path = next(iter(uploaded))

# 2. Init model & I/O
model = YOLO('runs/traffic/yolov8n_traffic2/weights/best.pt')
cap   = cv2.VideoCapture(video_path)
fps   = cap.get(cv2.CAP_PROP_FPS)
w, h  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out   = cv2.VideoWriter('output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
CLASSES = traffic_spec['names']

# 3. Process frames
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    results = model(frame)[0]
    for box in results.boxes:
        cid = int(box.cls[0]); name = model.names[cid]
        if name in CLASSES:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            label = f"{name} {conf:.2f}"
            cv2.rectangle(frame, (x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame, label, (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),2)
    out.write(frame)

cap.release(); out.release()

# 4. Display result
mp4 = open('output.mp4','rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f"""
<video width=640 controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")


In [8]:
#zip the runs folder

!zip -r runs.zip runs


  adding: runs/ (stored 0%)
  adding: runs/traffic/ (stored 0%)
  adding: runs/traffic/yolov8n_traffic2/ (stored 0%)
  adding: runs/traffic/yolov8n_traffic2/args.yaml (deflated 53%)
  adding: runs/traffic/yolov8n_traffic2/results.csv (deflated 58%)
  adding: runs/traffic/yolov8n_traffic2/confusion_matrix.png (deflated 21%)
  adding: runs/traffic/yolov8n_traffic2/confusion_matrix_normalized.png (deflated 18%)
  adding: runs/traffic/yolov8n_traffic2/train_batch1.jpg (deflated 9%)
  adding: runs/traffic/yolov8n_traffic2/val_batch1_labels.jpg (deflated 6%)
  adding: runs/traffic/yolov8n_traffic2/weights/ (stored 0%)
  adding: runs/traffic/yolov8n_traffic2/weights/best.pt (deflated 9%)
  adding: runs/traffic/yolov8n_traffic2/weights/last.pt (deflated 9%)
  adding: runs/traffic/yolov8n_traffic2/val_batch2_labels.jpg (deflated 6%)
  adding: runs/traffic/yolov8n_traffic2/P_curve.png (deflated 7%)
  adding: runs/traffic/yolov8n_traffic2/val_batch0_labels.jpg (deflated 10%)
  adding: runs/traffi